# ⚖️ 司律評分模型微調訓練 (Fine-Tuning Law Exam Evaluator)
使用 **Unsloth / LLaMA-Factory** 在 Google Colab (免費 T4 或 A100 GPU) 上微調開源模型（例如 Qwen2.5-7B 或 Taiwan-LLM）。
訓練資料直接讀取您的 Google Drive (`G:/我的雲端硬碟/LawExamData/training_data`)。

In [ ]:
# 1. 掛載 Google 雲端硬碟
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/LawExamData/training_data/law_exam_sft_dataset.json'
print('資料集路徑：', DATA_PATH)

In [ ]:
# 2. 安裝極速微調框架 Unsloth
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install xformers

In [ ]:
# 3. 載入基底開源模型 (支援 4-bit 快速微調)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# 選擇繁中與邏輯推理能力強大的 Qwen2.5-7B
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 啟用 LoRA 微調
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. 載入資料集並開始訓練
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = load_dataset('json', data_files=DATA_PATH, split='train')

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
# 5. 儲存 LoRA 模型權重回 Google Drive
SAVE_DIR = '/content/drive/MyDrive/LawExamData/lora_model'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('LoRA 權重已儲存至 Google Drive：', SAVE_DIR)